In [1]:
import os
import json
import pickle
import base64
from pathlib import Path
from typing import List
from dotenv import load_dotenv

# Unstructured for document parsing
from unstructured.partition.pptx import partition_pptx
from unstructured.chunking.title import chunk_by_title
from unstructured.documents.elements import Element

# LangChain components
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Load environment variables
load_dotenv()

d:\Projects_Main\ChunkSmith\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: 'NoneType' object is not subscriptable

In [2]:
!uv add "unstructured[pptx]"

Resolved 198 packages in 357ms
Uninstalled 30 packages in 2.07s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
error: Failed to install: psutil-7.2.1-cp37-abi3-win_amd64.whl (psutil==7.2.1)
  Caused by: failed to copy file from C:\Users\KAIZEN\AppData\Local\uv\cache\archive-v0\pJDObPhmCBN62TiK_KImb\psutil\_psutil_windows.pyd to D:\Projects_Main\ChunkSmith\.venv\Lib\site-packages\psutil\_psutil_windows.pyd: The process cannot access the file because it is being used by another process. (os error 32)


In [2]:
import os
from pathlib import Path
from typing import List

from pptx import Presentation
from unstructured.partition.pptx import partition_pptx

def extract_images_from_pptx(pptx_path: str, output_dir: str):
    # ✅ Ensure output directory exists (REAL FIX)
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    print("📁 Image output directory:", output_dir)
    print("📁 Exists:", os.path.exists(output_dir))

    prs = Presentation(pptx_path)
    image_count = 0

    for slide_idx, slide in enumerate(prs.slides):
        for shape_idx, shape in enumerate(slide.shapes):
            # MSO_SHAPE_TYPE.PICTURE == 13
            if shape.shape_type == 13:
                image = shape.image
                ext = image.ext  # png / jpeg

                image_name = f"slide_{slide_idx+1}_img_{shape_idx+1}.{ext}"
                image_path = os.path.join(output_dir, image_name)

                with open(image_path, "wb") as f:
                    f.write(image.blob)

                print(f"✅ Saved image: {image_path}")
                print("   Exists:", os.path.exists(image_path))
                print("   Size:", os.path.getsize(image_path))

                image_count += 1

    print(f"🖼️  Manually extracted {image_count} images from PPTX")


In [12]:
import os
from pathlib import Path
from typing import List
from unstructured.partition.pptx import partition_pptx

def partition_document_launcher(
    file_path: str,
    max_characters: int,
    new_after_n_chars: int,
    combine_text_under_n_chars: int,
    extract_images: bool = False,
    extract_tables: bool = False,
    languages: List[str] = ["eng"]
):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    if max_characters >= new_after_n_chars:
        raise ValueError("max_characters must be less than new_after_n_chars")

    # ✅ USE A REAL, SIMPLE PATH (NO GUESSING)
    image_output_dir = r"D:\Projects_Main\ChunkSmith\extracted_images"

    if extract_images:
        Path(image_output_dir).mkdir(parents=True, exist_ok=True)

    print(f"📄 Partitioning PPTX: {file_path}")
    print(f"⚙️  Images={extract_images}, Tables={extract_tables}")
    print(f"📁 Image dir: {image_output_dir}")

    # ---------- TEXT / TABLE EXTRACTION ----------
    elements = partition_pptx(
        filename=file_path,
        include_slide_notes=True,
        include_page_breaks=True,
        infer_table_structure=True,
        starting_page_number=1,
        strategy="hi_res",
        hi_res_model_name="yolox",
        chunking_strategy="by_title",
        include_orig_elements=True,
        languages=languages,
        max_characters=max_characters,
        new_after_n_chars=new_after_n_chars,
        combine_text_under_n_chars=combine_text_under_n_chars,
    )

    # print(f"✅ Extracted {len(elements)} text/table elements")

    # # ---------- IMAGE EXTRACTION ----------
    # if extract_images:
    #     extract_images_from_pptx(
    #         pptx_path=file_path,
    #         output_dir=image_output_dir
    #     )

    # # ---------- ELEMENT BREAKDOWN ----------
    # element_types = {}
    # for elem in elements:
    #     name = type(elem).__name__
    #     element_types[name] = element_types.get(name, 0) + 1

    # print(f"📋 Element breakdown: {element_types}")

    return elements


In [13]:
checkpoint1 = partition_document_launcher (file_path =r"D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx",
                                          max_characters=3000,
                                          new_after_n_chars=3800,
                                          combine_text_under_n_chars=200,
                                          extract_images=True,
                                          extract_tables=True,
                                          languages=['eng'],            
                                          )

📄 Partitioning PPTX: D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx
⚙️  Images=True, Tables=True
📁 Image dir: D:\Projects_Main\ChunkSmith\extracted_images


In [18]:
checkpoint1[0].metadata.orig_elements[0].to_dict()

{'type': 'NarrativeText',
 'element_id': '644c15d1-e402-4285-bc6c-1e134740d221',
 'text': '1',
 'metadata': {'category_depth': 0,
  'file_directory': 'D:\\Projects_Main\\ChunkSmith\\docs\\pdf',
  'filename': 'Philips-Innovation-That-Matters.pptx.pptx',
  'last_modified': '2025-12-30T19:17:56',
  'page_number': 1}}